In [8]:
from modules import MLPImputer

import torch

from tqdm.auto import tqdm

from torch.utils.data import TensorDataset, DataLoader

In [9]:
N_COLUMNS = 5
TRAIN_SIZE = 100000
VALID_SIZE = 1000

CAT_FEATURES = torch.tensor([1])
CLASSES_COUNT = torch.tensor([3])

BATCH_SIZE = 10000
NUM_EPOCHS = 100

In [10]:
def generate_task():
    train_mask = torch.randint(0, 2, size=(TRAIN_SIZE, N_COLUMNS), dtype=torch.bool)
    train_data = torch.randn(TRAIN_SIZE, N_COLUMNS)

    for col, cnt in zip(CAT_FEATURES, CLASSES_COUNT):
        train_data[:, col] = torch.randint(0, cnt, size=(TRAIN_SIZE,))

    train_data *= train_mask
    
    train_dataset = TensorDataset(train_data, train_mask)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,)
    
    valid_mask = torch.randint(0, 2, size=(VALID_SIZE, N_COLUMNS), dtype=torch.bool)
    valid_data = torch.randn(VALID_SIZE, N_COLUMNS)

    for col, cnt in zip(CAT_FEATURES, CLASSES_COUNT):
        valid_data[:, col] = torch.randint(0, cnt, size=(VALID_SIZE,))
    
    valid_dataset = TensorDataset(valid_data, valid_mask)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    return train_loader, valid_loader

In [11]:
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

In [12]:
train_loader, valid_loader = generate_task()

In [13]:
model = MLPImputer(
    n_columns=N_COLUMNS,
    d_model=256,
    embed_dim=256,
    encoder_layers=32,
    encoder_output_dim=32,
    head_layers=32,
    dropout=0.1
)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, NUM_EPOCHS // 2)

pbar_epoch = tqdm(range(NUM_EPOCHS), desc="Epoch", leave=False)
for _ in pbar_epoch:

    model.train()
    pbar_train = tqdm(train_loader, desc="Training", leave=False)
    for data, mask in pbar_train:
        data_out, mask_out, dropped = MLPImputer.transform(data, mask, drop_rate=0.5)

        optimizer.zero_grad()
        output = model.forward(data_out, mask_out)
        loss = model.loss(output, data, dropped)
        loss.backward()

        optimizer.step()

        scheduler.step()

        pbar_train.set_postfix(train_loss=f"{loss.item():.4f}",
                               lr=f"{optimizer.param_groups[0]["lr"]:.4f}")

    valid_loss = 0.0
    valid_steps = 0
    model.eval()
    with torch.no_grad():
        for data, mask in tqdm(valid_loader, desc="Validation", leave=False):
            data_forward = data * mask
            output = model.forward(data_forward, mask)
            valid_loss += model.loss(output, data, ~mask).item()
            valid_steps += 1

    pbar_epoch.set_postfix(valid_loss=f"{valid_loss / valid_steps:.4f}")


    

Epoch:   0%|          | 0/100 [00:00<?, ?it/s]

Training:   0%|          | 0/10 [00:00<?, ?it/s]

Validation:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/10 [00:00<?, ?it/s]

Validation:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/10 [00:00<?, ?it/s]

Validation:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/10 [00:00<?, ?it/s]